In [ ]:
import string
import os, sys
import pickle
import re
import random
import cv2
from datetime import timedelta, datetime
import matplotlib.dates as mdates

import numpy as np
import pandas as pd
import tensorflow as tf
import gc
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.losses import MeanAbsoluteError, MeanSquaredError
from scipy.stats import pearsonr

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from astropy.io import fits

try:
    current_folder = os.getcwd()
    hsemetric_folder = os.path.abspath(os.path.join(current_folder, "../HSEMETRIC"))
    sys.path.append(hsemetric_folder)
    import hsemetric
except Exception:
    hsemetric = None
    print("hsemetric not available — Threat_score will be set to NaN.")

def reduced_chi_squared(y_true, y_pred, sigma):
    """
    Calculate the reduced chi-squared (reduced mean square error)
    
    Parameters:
    ----------
    y_true : array-like : True observed values
    y_pred : array-like : Predicted values
    sigma  : array-like : Standard deviation (errors) of the true values
    
    Returns:
    ----------
    float : Reduced chi-squared value

    Examples:
    ---------
    >>> reduced_chi_squared(y_true, 
                           y_pred,
                           sigma, 
        )
    """
    
    if len(y_true) != len(y_pred) or len(y_true) != len(sigma):
        raise ValueError("Input arrays must have the same length")

    # Calculate the reduced chi-squared
    chi_squared = np.sum(((y_true - y_pred) ** 2) / (sigma ** 2))
    reduced_chi_squared = chi_squared / len(y_true)
    
    return reduced_chi_squared

def prepare_dataset(sw_csv, icme_excel, img_path, std_csv):
    """
    Load and merge solar wind speed, standard deviation, and ICME list.

    Parameters
    ----------
    sw_csv : str
        Path to solar wind CSV file.
    icme_excel : str
        Path to ICME Excel file.
    img_path : str
        Directory containing EUV image files.
    std_csv : str
        Path to CSV containing standard deviation values.

    Returns
    -------
    tuple
        (merged dataframe, dict of date->image_file, icme_dates set)
    """
    df_sw = pd.read_csv(sw_csv)
    df_sw["DATE"] = pd.to_datetime(df_sw["DATE"])
    df_sw = df_sw.rename(columns={"V(Km/s)": "speed"})

    df_std = pd.read_csv(std_csv)
    df_std["DATE"] = pd.to_datetime(df_std["DATE"])
    df_std = df_std.rename(columns={"V(Km/s)": "std_speed"})

    df = pd.merge(df_sw, df_std, on="DATE", how="inner")

    icme_list = pd.read_excel(icme_excel)
    icme_list["Start_time"] = pd.to_datetime(icme_list["Start_time"].str.split(" ").str[0], format="%Y/%m/%d")
    icme_list["End_time"] = pd.to_datetime(icme_list["End_time"].str.split(" ").str[0], format="%Y/%m/%d")

    icme_dates = []
    for start, end in zip(icme_list["Start_time"], icme_list["End_time"]):
        icme_dates.extend(pd.date_range(start, end).tolist())
    icme_dates = set(icme_dates)

    df = df[~df["DATE"].isin(icme_dates)].reset_index(drop=True)

    img_files, img_dates = [], []
    for f in os.listdir(img_path):
        if f.isdigit() and len(f) == 8:
            date = pd.to_datetime(f, format="%Y%m%d")
        else:
            m = re.search(r"(\d{8})", f)
            if not m:
                continue
            date = pd.to_datetime(m.group(1), format="%Y%m%d")

        if date in icme_dates:
            continue
        img_files.append(f)
        img_dates.append(date)

    img_dates, img_files = zip(*sorted(zip(img_dates, img_files)))
    return df, dict(zip(img_dates, img_files)), icme_dates


def match_data(df_sw, img_dict, delay):
    """
    Match solar wind + std to EUV images with forecast delay.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing DATE, speed, std_speed.
    img_dict : dict
        Mapping from date -> image filename.
    delay : int
        Forecast delay (days).

    Returns
    -------
    pd.DataFrame
        Matched dataset with sw_date, img_date, file, speed, std_speed.
    """
    matched = []
    for _, row in df_sw.iterrows():
        sw_date, sw_speed, std_speed = row["DATE"], row["speed"], row["std_speed"]
        img_date = sw_date - timedelta(days=delay)
        if img_date in img_dict:
            matched.append((sw_date, img_date, img_dict[img_date], sw_speed, std_speed))
    return pd.DataFrame(matched, columns=["sw_date", "img_date", "img_file", "speed", "std_speed"])


# -------------------------
# Batch splitting (consecutive by img_date)
# -------------------------
def split_batches(df, batch_size=20):
    batches, i = [], 0
    while i + batch_size <= len(df):
        window = df.iloc[i:i+batch_size]
        if (window["img_date"].diff().dropna() == pd.Timedelta(days=1)).all():
            batches.append(window)
            i += batch_size
        else:
            i += 1
    return batches


# -------------------------
# Random split: Train+Val batches vs Test batches (fixed test for all folds)
# -------------------------
def split_train_val_test(df_matched, test_fraction=0.2, n_splits=5, seed=42, batch_size=20):
    np.random.seed(seed)
    all_batches = split_batches(df_matched, batch_size=batch_size)
    num_batches = len(all_batches)
    n_test = max(1, int(test_fraction * num_batches))
    test_indices = np.random.choice(num_batches, size=n_test, replace=False)

    test_batches = [all_batches[i] for i in test_indices]
    train_val_batches = [b for i, b in enumerate(all_batches) if i not in test_indices]

    print(f"Total batches: {num_batches}, Test batches: {len(test_batches)}, Train+Val batches: {len(train_val_batches)}")
    return train_val_batches, test_batches

# -------------------------
# Load map file (FITS -> np.array, HxW[xC])
# -------------------------
def load_map(filepath):
    with fits.open(filepath) as fit_img:
        arr = np.array(fit_img[0].data, dtype=np.float32)
    if arr.ndim == 2:
        arr = np.expand_dims(arr, -1)
    return arr

def compute_metrics(y_true, y_pred, y_std, hsemetric=None):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mae = MeanAbsoluteError()(y_true, y_pred).numpy()
    mse = MeanSquaredError()(y_true, y_pred).numpy()
    rmse = float(np.sqrt(mse))
    n = len(y_true)

    # --- Uncertainty of MSE
    errors = y_true - y_pred
    sq_errors = errors ** 2
    mse_std = np.std(sq_errors, ddof=1) / np.sqrt(n)

    # --- Uncertainty of RMSE (propagation)
    rmse_std = (0.5 / rmse) * mse_std if rmse > 0 else np.nan

    # guard for tiny arrays
    if len(y_true) > 1 and len(y_pred) > 1:
        corr, p_value = pearsonr(y_true, y_pred)
    else:
        corr, p_value = (np.nan, np.nan)
    
    if abs(corr) < 1 and n > 3:
        sigma_z = 1 / np.sqrt(n - 3)
        corr_std = (1 - corr**2) * sigma_z
    else:
        corr_std = np.nan
        
    # threat score
    
    if hsemetric is not None:
        try:
            tp, fp, fn = hsemetric.BatchwiseHSE(y_true, np.array(y_pred))
            threat_score = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else np.nan
            bias = (tp + fn) / (tp + fp) if (tp + fp) > 0 else np.nan
            ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
            far = fp / (tp + fp) if (tp + fp) > 0 else np.nan
        except Exception:
            threat_score = bias = ppv = far = np.nan
    else:
        threat_score = bias = ppv = far = np.nan

    reduced_chi_squ = reduced_chi_squared(y_true, np.array(y_pred), np.array(y_std))

    return dict(MAE=float(mae), MSE=float(mse), MSE_error = float(mse_std), RMSE=float(rmse), 
                RMSE_error = float(rmse_std), Corr=float(corr),
                Corr_std=float(corr_std), P_Value=float(p_value), Threat_Score=float(threat_score),
                Bias=float(bias), PPV=float(ppv), FAR=float(far), Reduced_Chi_Squared = float(reduced_chi_squ))


# -------------------------
# Persistence baselines
# -------------------------
def persistence_27_on_dates(df_sw, sw_dates, period_days=27.0, fill_method=None):
    """
    Classic persistence: predicted(t) = V(t - period_days)
    """
    s = (df_sw[["DATE", "speed"]]
         .dropna()
         .sort_values("DATE")
         .set_index("DATE")["speed"]
         .asfreq("1D"))
    if fill_method in ("ffill", "bfill"):
        s = getattr(s, fill_method)()

    lag = pd.Timedelta(days=period_days)
    preds = []
    for d in sw_dates:
        prev = d - lag
        val = float(s.loc[prev]) if prev in s.index else np.nan
        preds.append(val)
    return np.array(preds, dtype=float)


def persistence_N_on_dates(df_sw, sw_dates, delay, period_days=27.0, fill_method=None):
    """
    N-day persistence WITH delay:
        predicted(t) = V(t + delay - period_days)

    Examples:
        - If delay=4 and N=27: predicted(t) = V(t + 4 - 27).
        - Set period_days to any N you want.
    """
    s = (df_sw[["DATE", "speed"]]
         .dropna()
         .sort_values("DATE")
         .set_index("DATE")["speed"]
         .asfreq("1D"))
    if fill_method in ("ffill", "bfill"):
        s = getattr(s, fill_method)()

    lag = pd.Timedelta(days=period_days)
    preds = []
    for d in sw_dates:
        prev = d + timedelta(days=delay) - lag
        val = float(s.loc[prev]) if prev in s.index else np.nan
        preds.append(val)
    return np.array(preds, dtype=float)



# ========================
# Full SC plotting
# ========================

# ---- Helpers ----
def compute_corr(x, y):
    mask = ~np.isnan(x) & ~np.isnan(y)
    if mask.sum() > 1:
        r, p = pearsonr(x[mask], y[mask])
        return r, p
    return np.nan, np.nan


def compute_rmse(obs, pred):
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    if mask.sum() > 0:
        y_true = obs[mask]
        y_pred = pred[mask]
        mse = MeanSquaredError()(y_true, y_pred).numpy()
        rmse = float(np.sqrt(mse))
        
        n = len(y_true)
        # --- Uncertainty of MSE
        errors = y_true - y_pred
        sq_errors = errors ** 2
        mse_std = np.std(sq_errors, ddof=1) / np.sqrt(n)

        # --- Uncertainty of RMSE (propagation)
        rmse_std = (0.5 / rmse) * mse_std if rmse > 0 else np.nan
        return rmse, rmse_std
    return np.nan, np.nan


def add_icme_lines(ax, sw_dates, icme_dates):
    for i, icme_date in enumerate(icme_dates):
        if sw_dates.min() <= icme_date <= sw_dates.max():
            ax.axvline(icme_date, color="black", linestyle="-",
                       ymax=1, ymin=0.95, linewidth=2.5,
                       label="ICME Event" if i == 0 else "")

       
def add_hse_verticals(ax, sw_dates, y_obs, dat, color="brown"):
    _, time_final, _ = hsemetric.HSE(y_obs)
    for k, (start_idx, end_idx) in enumerate(time_final):
        if 0 <= start_idx < len(sw_dates) and 0 <= end_idx < len(sw_dates):
            ax.axvspan(sw_dates[start_idx], sw_dates[end_idx],
                       color=color, alpha=0.3,
                       label=f"{dat} HSE Event" if k == 0 else "")
    ax.legend(loc="upper left")




def plot_with_gaps(ax, x, y, color, label, marker, gap_tolerance, linewidth=1.0, markersize=3):
    """
    Helper to plot time series with NaN gaps preserved as breaks.
    """
    seg_x, seg_y = [], []
    for i in range(len(x)):
        if np.isnan(y[i]):
            if seg_x:
                ax.plot(seg_x, seg_y, color=color, linewidth=linewidth,
                        marker=marker, markersize=markersize,
                        label=label if label else None)
                label = None
                seg_x, seg_y = [], []
            continue
        if seg_x and (x[i] - seg_x[-1]) > gap_tolerance:
            ax.plot(seg_x, seg_y, color=color, linewidth=linewidth,
                    marker=marker, markersize=markersize,
                    label=label if label else None)
            label = None
            seg_x, seg_y = [], []
        seg_x.append(x[i]); seg_y.append(y[i])
    if seg_x:
        ax.plot(seg_x, seg_y, color=color, linewidth=linewidth,
                marker=marker, markersize=markersize,
                label=label if label else None)


def build_gap_index(sw_dates, y_obs, y_model, y_pers27, y_persN, max_nans=5, gap_size=3):
    """
    Compress long NaN runs and build artificial index with gaps.
    """
    df_plot = pd.DataFrame({
        "sw_date": sw_dates,
        "observed": y_obs.reshape(-1),
        "model": y_model.reshape(-1),
        "pers27": y_pers27.reshape(-1),
        "persN": y_persN.reshape(-1)
    }).sort_values("sw_date").reset_index(drop=True)

    # --- Compress long NaN runs ---
    compressed_rows = []
    nan_count = 0
    for _, row in df_plot.iterrows():
        if pd.isna(row["observed"]) and pd.isna(row["model"]) and pd.isna(row["pers27"]) and pd.isna(row["persN"]):
            nan_count += 1
            if nan_count <= max_nans:
                compressed_rows.append(row)
        else:
            nan_count = 0
            compressed_rows.append(row)
    df_plot = pd.DataFrame(compressed_rows).reset_index(drop=True)

    # --- Build artificial index with spacing for gaps ---
    index_with_gaps = []
    current_idx = 0
    prev_date = None
    for d in df_plot["sw_date"]:
        if prev_date is not None and (d - prev_date).days > 1:
            current_idx += gap_size
        index_with_gaps.append(current_idx)
        current_idx += 1
        prev_date = d
    df_plot["plot_idx"] = index_with_gaps

    return df_plot



    
def plot_fullsc(sw_dates, y_obs, y_model, y_pers27, y_persN,
                *, delay, fold_index, corr_m, p_m, save_path,
                title_prefix="SpeedNet", model_type="EUV", three_column=True, gap_size=3, max_nans=5):

    """
    Plot with either 2 or 3 columns:
      - Col 1: Obs + SpeedNet + 27-day + N-day (with gaps).
      - Col 2 (optional, wider): Obs + SpeedNet only.
      - Col 3: Correlation scatter.
    """

    # --- Prepare Data ---
    df_plot = build_gap_index(sw_dates, y_obs, y_model, y_pers27, y_persN,
                              max_nans=max_nans, gap_size=gap_size)

    idx   = df_plot["plot_idx"].values
    dates = df_plot["sw_date"].values
    obs   = df_plot["observed"].values
    pred  = df_plot["model"].values
    pers27 = df_plot["pers27"].values
    persN  = df_plot["persN"].values

    # --- Layout ---
    if three_column:
        fig = plt.figure(figsize=(36, 7))
        gs = GridSpec(1, 3, figure=fig, width_ratios=[8, 12, 4], wspace=0.3)
        ax_all  = fig.add_subplot(gs[0, 0])
        ax_pair = fig.add_subplot(gs[0, 1])
        ax_corr = fig.add_subplot(gs[0, 2])
    else:
        fig = plt.figure(figsize=(28, 7))
        gs = GridSpec(1, 2, figure=fig, width_ratios=[12, 4], wspace=0.3)
        ax_all  = fig.add_subplot(gs[0, 0])
        ax_corr = fig.add_subplot(gs[0, 1])
        ax_pair = None  # not used

    fig.suptitle(f"{title_prefix}-{model_type} [Full Cycle], Delay={delay}",
                 fontsize=16, fontweight="bold", y=1.02)

    panel_labels = ["(a)", "(b)", "(c)"]
    
    for label, ax in zip(panel_labels, [ax_all, ax_pair if ax_pair else None, ax_corr]):
        if ax is not None:
            ax.annotate(label, xy=(0, 1.05), xycoords="axes fraction",
                        fontsize=16, fontweight="bold",
                        ha="left", va="bottom")
    
    # --- Column 1: All series ---
    plot_with_gaps(ax_all, idx, pred,   color="blue",   label=f"SpeedNet-{model_type}", marker="o", gap_tolerance=gap_size)
    plot_with_gaps(ax_all, idx, pers27, color="red",    label="27-day",      marker="s", gap_tolerance=gap_size)
    plot_with_gaps(ax_all, idx, persN,  color="orange", label="N-day",       marker="^", gap_tolerance=gap_size)
    plot_with_gaps(ax_all, idx, obs,    color="green",  label="Observed",    marker=".", gap_tolerance=gap_size)
    ax_all.set_ylabel("V (Km/s)", fontsize=12, fontweight="bold")
    ax_all.set_xlabel("Date", fontsize=12, fontweight="bold")
    ax_all.legend(loc="upper left")
    step = max(1, len(idx)//15)
    tick_locs = idx[::step]
    tick_labels = [pd.to_datetime(dates[i]).strftime("%Y-%m-%d") for i in range(0, len(dates), step)]
    ax_all.set_xticks(tick_locs)
    ax_all.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)

    # --- Column 2: Obs vs SpeedNet only (optional) ---
    if ax_pair is not None:
        plot_with_gaps(ax_pair, idx, pred, color="blue", label=f"SpeedNet-{model_type}", marker="o", gap_tolerance=gap_size)
        plot_with_gaps(ax_pair, idx, obs,  color="green", label="Observed",   marker=".", gap_tolerance=gap_size)
        ax_pair.set_ylabel("V (Km/s)", fontsize=12, fontweight="bold")
        ax_pair.set_xlabel("Date", fontsize=12, fontweight="bold")
        ax_pair.legend(loc="upper left")
        ax_pair.set_xticks(tick_locs)
        ax_pair.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)

        # add_hse_verticals(ax_pair, idx, obs)
        add_hse_verticals(ax_pair, idx, obs, dat=f'Observed', color="brown")
        add_hse_verticals(ax_pair, idx, pred, dat=f'SpeedNet-{model_type}', color="blue")
        # add_hse_verticals(ax_pair, dates, obs)
        
        
        tp, fp, fn = hsemetric.BatchwiseHSE(y_obs, np.array(y_model))
        ts = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else np.nan
        rmse, err = compute_rmse(y_obs, y_model)

        ax_pair.text(0.98, 0.95,
                f"r={corr_m:.2f}, TS={ts:.2f}\nRMSE={rmse:.1f}±{err:.1f}",
                transform=ax_pair.transAxes, fontsize=7, fontweight="bold",
                ha="right", va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.6))


    # --- Column 3: Correlation scatter ---
    mask_model  = ~np.isnan(obs) & ~np.isnan(pred)
    mask_pers27 = ~np.isnan(obs) & ~np.isnan(pers27)
    mask_persN  = ~np.isnan(obs) & ~np.isnan(persN)

    ax_corr.scatter(obs[mask_model],  pred[mask_model],    alpha=0.6, s=14, c="blue",   label=f"Obs vs SpeedNet-{model_type}")
    ax_corr.scatter(obs[mask_pers27], pers27[mask_pers27], alpha=0.6, s=14, c="red",    label="Obs vs 27-day")
    ax_corr.scatter(obs[mask_persN],  persN[mask_persN],   alpha=0.6, s=14, c="orange", label="Obs vs N-day")

    if mask_model.sum() > 1:
        m_slope, m_inter = np.polyfit(obs[mask_model], pred[mask_model], 1)
        ax_corr.plot(obs[mask_model], m_slope*obs[mask_model] + m_inter, 'b--', linewidth=1.0)
    if mask_pers27.sum() > 1:
        p27_slope, p27_inter = np.polyfit(obs[mask_pers27], pers27[mask_pers27], 1)
        ax_corr.plot(obs[mask_pers27], p27_slope*obs[mask_pers27] + p27_inter, 'r--', linewidth=1.0)
    if mask_persN.sum() > 1:
        pN_slope, pN_inter = np.polyfit(obs[mask_persN], persN[mask_persN], 1)
        ax_corr.plot(obs[mask_persN], pN_slope*obs[mask_persN] + pN_inter, 'orange', linestyle="--", linewidth=1.0)

    ax_corr.set_title("Correlation", fontsize=14, fontweight="bold")
    ax_corr.set_xlabel("Observed V (Km/s)", fontsize=12, fontweight="bold")
    ax_corr.set_ylabel("Predicted V (Km/s)", fontsize=12, fontweight="bold")
    
    r_27, p_27 = compute_corr(obs[mask_pers27], pers27[mask_pers27])
    r_N, p_N = compute_corr(obs[mask_persN], persN[mask_persN])
    ax_corr.text(0.98, 0.04,
                f"SpeedNet-{model_type}: r={corr_m:.2f}, p={p_m:.1e}\n"
                f"27-day: r={r_27:.2f}, p={p_27:.1e}\n"
                f"N-day: r={r_N:.2f}, p={p_N:.1e}",
                transform=ax_corr.transAxes, fontsize=12,
                verticalalignment="bottom", horizontalalignment="right",
                bbox=dict(boxstyle="round", facecolor="white"))

    ax_corr.legend(loc="upper left")

        # --- Save with suffix depending on type ---
    base, ext = os.path.splitext(save_path)
    if three_column:
        save_path_out = f"{base}_3col{ext}"
        save_path_pdf = f"{base}_{model_type}.pdf"
    else:
        save_path_out = f"{base}_2col{ext}"
        save_path_pdf = f"{base}_{model_type}.pdf"

    os.makedirs(os.path.dirname(save_path_out), exist_ok=True)
    plt.savefig(save_path_out, dpi = 500, bbox_inches="tight")
    plt.savefig(save_path_pdf, dpi = 500, bbox_inches="tight")
    plt.close()
    print(f"Saved plot → {save_path_out}")

# ========================
# Attribution maps
# ========================
@tf.custom_gradient
def guided_relu(x):
    def grad(dy):
        return tf.cast(dy > 0, tf.float32) * tf.cast(x > 0, tf.float32) * dy
    return tf.nn.relu(x), grad


def replace_relu_with_guided(model):
    def convert(layer):
        if hasattr(layer, "activation") and layer.activation == tf.keras.activations.relu:
            config = layer.get_config()
            config["activation"] = guided_relu
            return layer.__class__.from_config(config)
        return layer

    guided_model = tf.keras.models.clone_model(model, clone_function=convert)
    guided_model.set_weights(model.get_weights())
    return guided_model


def explain_image_with_heatmaps(img_tensor, best_model, last_conv_layer_name="max_pooling2d_3", class_index=None, baseline=None):
    if isinstance(img_tensor, np.ndarray):
        img_tensor = tf.convert_to_tensor(img_tensor, dtype=tf.float32)
    if len(img_tensor.shape) == 3:
        img_tensor = tf.expand_dims(img_tensor, axis=0)

    # -------- GRAD-CAM --------
    grad_model = Model(
        inputs=best_model.inputs,
        outputs=[best_model.get_layer(last_conv_layer_name).output, best_model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        if class_index is None:
            class_index = tf.argmax(predictions[0])
        loss = predictions[:, class_index]

    grads = tape.gradient(loss, conv_outputs)[0]  # Shape: (H, W, C)
    conv_outputs = conv_outputs[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))

    conv_outputs = conv_outputs * pooled_grads
    heatmap = tf.reduce_sum(conv_outputs, axis=-1)
    heatmap = tf.maximum(heatmap, 0).numpy()
    heatmap = cv2.resize(heatmap, (img_tensor.shape[2], img_tensor.shape[1]))

    # -------- GUIDED GRAD-CAM --------
    # guided_model = replace_relu_with_guided(best_model)

    # with tf.GradientTape() as tape:
    #     tape.watch(img_tensor)
    #     out = guided_model(img_tensor)
    #     loss = out[:, class_index]

    # guided_grads = tape.gradient(loss, img_tensor)[0].numpy()
    # guided_grads = cv2.resize(guided_grads, (img_tensor.shape[2], img_tensor.shape[1]))
    # guided_gradcam = guided_grads * heatmap[..., np.newaxis]
    # # guided_gradcam = guided_grads * heatmap

    # -------- INTEGRATED GRADIENTS --------
    def interpolate_images(baseline, image, alphas):
        return baseline + alphas * (image - baseline)

    def compute_integrated_gradients(image, baseline, model, class_index, steps=50):
        alphas = tf.linspace(0.0, 1.0, steps)
        alphas_x = alphas[:, tf.newaxis, tf.newaxis, tf.newaxis]
        interpolated = interpolate_images(baseline, image, alphas_x)
        interpolated = tf.reshape(interpolated, (steps,) + image.shape[1:])
        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            pred = model(interpolated)
            selected_output = pred[:, class_index]
        grads = tape.gradient(selected_output, interpolated)
        avg_grads = tf.reduce_mean(grads, axis=0)
        integrated_gradients = (image - baseline)[0] * avg_grads
        return integrated_gradients.numpy()

    ### I CHANGE THE BASELINE FROM JUST ZERO-LIKE
    # if baseline is None:
    #     baseline = tf.zeros_like(img_tensor)
    # integrated_grad = compute_integrated_gradients(img_tensor, baseline, best_model, int(class_index))
    
    ### TO RANDOM RESAMPLING
    if baseline is None:
        # Create shuffled baseline
        flat = img_tensor.numpy().reshape(-1, img_tensor.shape[-1])  # flatten [num_pixels, channels]
        np.random.shuffle(flat)  # shuffle pixel rows
        shuffled_img = flat.reshape(img_tensor.shape)
        baseline = tf.convert_to_tensor(shuffled_img, dtype=img_tensor.dtype)

    # Compute integrated gradients
    integrated_grad = compute_integrated_gradients(img_tensor, baseline, best_model, int(class_index))
    
    return heatmap, integrated_grad, predictions.numpy()

    # return heatmap, guided_gradcam, integrated_grad, predictions.numpy()


def save_individual_attributions(X_test, sw_test_dates, delay, model, phase, base_save_dir):
    """
    Save individual attribution maps (Grad-CAM, Guided Grad-CAM, IG) as separate .npy files.
    Organized into subfolders by type.
    
    Args:
        X_test: Numpy array of test images
        sw_test_dates: List of dates or identifiers matching each sample
        delay: Integer or string to append to file name
        model: The trained model to use
        phase: Name of the phase (used for folder structure)
        base_save_dir: Root directory where results will be saved
    """
    test_set_dir_per_delay = os.path.join(base_save_dir, "Attribution", f"Test_Set_delay_{delay}")
    gradcam_dir = os.path.join(base_save_dir, "Attribution", "Gradcam")
    # guided_dir = os.path.join(base_save_dir, "Attribution", "GuidedGradcam")
    ig_dir = os.path.join(base_save_dir, "Attribution", "IG")

    os.makedirs(test_set_dir_per_delay, exist_ok=True)
    os.makedirs(gradcam_dir, exist_ok=True)
    # os.makedirs(guided_dir, exist_ok=True)
    os.makedirs(ig_dir, exist_ok=True)

    for i in range(len(X_test)):
        img = X_test[i].reshape(1, *X_test.shape[1:])
        # gradcam, guided_gradcam, ig, _ = explain_image_with_heatmaps(img, model)
        gradcam, ig, _ = explain_image_with_heatmaps(img, model)

        # Subtract Delay days
        new_date = sw_test_dates[i] - timedelta(days=delay)

        # Format to YYYY-MM-DD
        new_date_str = new_date.strftime("%Y_%m_%d")
            
        filename = f"{new_date_str}.npy"

        np.save(os.path.join(gradcam_dir, filename), gradcam)
        # np.save(os.path.join(guided_dir, filename), guided_gradcam)
        np.save(os.path.join(ig_dir, filename), ig)
        np.save(os.path.join(test_set_dir_per_delay, filename), X_test[i])

        # Optional: cleanup per iteration
        K.clear_session()

# ========================
# Main entry
# ========================

def model_evaluate_and_persistence(sw_csv, std_csv, icme_excel, img_path, model_path,                              
                                delay, fold_index,
                                batch_size, lstm_unit, dense_unit,
                                seed=42, test_fraction=0.2, batch_window=20,
                                save_path="./results", hsemetric=None,
                                persistence_period_days=27.0, persistence_fill=None,
                                N_period_days=27.0, model_type="EUV"):
    """
    1) Builds the SAME test set (from split_train_val_test) for the given delay.
    2) Train the model on X_train and evaluate/test on X_test and compute metrics.
    3) Computes a 27-day persistence baseline on the SAME test dates and metrics.
    4) Plots Observed vs Predicted (Model + Persistence) and dual correlation.
    5) Saves artifacts under save_path and returns two DataFrames:
       - df_metrics_model
       - df_metrics_persistence
    """
    
    
    os.makedirs(save_path, exist_ok=True)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
    # ---- Prepare data ----
    # df_sw, img_dict, icme_dates = prepare_dataset(sw_csv, icme_excel, img_path)
    df_sw, img_dict, icme_dates = prepare_dataset(sw_csv, icme_excel, img_path, std_csv)
    print(f"Loaded {len(df_sw)} SW records and {len(img_dict)} images from {img_path}")

    
    n_splits = 5
   
    delays = [delay] if delay is not None else [1, 2, 3, 4, 5]

    for delay in delays:
        print(f"\n=== Delay {delay} ===")

        # ---- Match with images based on delay ----
        df_matched = match_data(df_sw, img_dict, delay)
        if df_matched.empty:
            raise ValueError("No matched SW/image records for the specified delay.")

        train_val_batches, test_batches = split_train_val_test(df_matched, test_fraction=test_fraction,
                                                               n_splits=n_splits, seed=seed)

        # n_splits = 5
        num_samples = len(train_val_batches)  # number of train+val batches

        for fold_id in range(n_splits):
            val_start = fold_id * (num_samples // n_splits)
            val_end = val_start + (num_samples // n_splits)
            if val_end > num_samples:
                val_end = num_samples
                
            if (fold_index is not None) and (fold_id != fold_index):
                continue

            print(f"Plotting during the Fold {fold_id} and Delay: {delay}")
            # Define train/validation batches
            train_idx = list(range(0, val_start)) + list(range(val_end, num_samples))
            val_idx = list(range(val_start, val_end))

            # Load train/val data
            X_train = np.array([load_map(os.path.join(img_path, r.img_file))
                                for i in train_idx for _, r in train_val_batches[i].iterrows()])
            y_train = np.array([r.speed for i in train_idx for _, r in train_val_batches[i].iterrows()])

            X_val = np.array([load_map(os.path.join(img_path, r.img_file))
                            for i in val_idx for _, r in train_val_batches[i].iterrows()])
            y_val = np.array([r.speed for i in val_idx for _, r in train_val_batches[i].iterrows()])

            # Test set remains fixed
            X_test = np.array([load_map(os.path.join(img_path, r.img_file))
                            for b in test_batches for _, r in b.iterrows()])
            y_test = np.array([r.speed for b in test_batches for _, r in b.iterrows()])
           
            std_test = np.array([r.std_speed for b in test_batches for _, r in b.iterrows()])
            
            sw_test_dates = np.array([r.sw_date for b in test_batches for _, r in b.iterrows()])
            
            print(f"Fold {fold_index}: Train {len(X_train)}, Val {len(X_val)}, Test {len(X_test)}")


            # input_shape = X_train.shape[1:]
            model_filename = (f"{model_path}/best_model_fold{fold_index}_delay{delay}_"
                            f"bs{batch_size}_lstm{lstm_unit}_dense{dense_unit}.keras")
            
            loaded = load_model(model_filename)
            preds_model = loaded.predict(X_test, verbose=1).reshape(-1)
            
            # Correlation
            corr_m, p_m = compute_corr(y_test, preds_model)

            # 27-Days Persistence
            preds_pers27 = persistence_27_on_dates(df_sw, sw_test_dates,
                                                period_days=persistence_period_days,
                                                fill_method=persistence_fill)
            
            # N-day Persistence
            preds_persN = persistence_N_on_dates(df_sw, sw_test_dates,
                                                delay=delay,
                                                period_days=N_period_days,
                                                fill_method=persistence_fill)
            
            # Plot Full SC
            df_prediction_speednet = pd.DataFrame({
                "Date": sw_test_dates,
                "Observed": y_test,
                "prediction": preds_model,
                "Observed_std": std_test,
                "27-day": preds_pers27,
                "N-day": preds_persN
            })

            
            # Ensure Date is datetime
            df_prediction_speednet["Date"] = pd.to_datetime(df_prediction_speednet["Date"])

            # Sort by Date
            df_prediction_speednet = df_prediction_speednet.sort_values("Date").reset_index(drop=True)

            # Save to Excel
            speednet_prediction_file = os.path.join(
                save_path, f"prediction_speednet_fold{fold_index}_delay{delay}.xlsx"
            )
            df_prediction_speednet.to_excel(speednet_prediction_file, index=False)


            # Folder for saving FullSC plot data
            save_dir = f"./SpeedNet_{model_type}_fullsc_data"
            os.makedirs(save_dir, exist_ok=True)

            # Dictionary of everything passed into plot_fullsc
            fullsc_data = {
                "sw_test_dates": sw_test_dates,
                "y_test": y_test,
                "preds_model": preds_model,
                "preds_pers27": preds_pers27,
                "preds_persN": preds_persN,
                "delay": delay,
                "fold_index": fold_index,
                "corr_m": corr_m,
                "p_m": p_m,
                "title_prefix": "SpeedNet",
                "model_type": model_type,
                "three_column": True,
                "gap_size": 3,
                "max_nans": 5
            }

            # Save all plot data to a pickle file
            file_path = os.path.join(save_dir, f"SpeedNet_{model_type}_FullSC_fold{fold_index}_delay{delay}.pkl")
            with open(file_path, "wb") as f:
                pickle.dump(fullsc_data, f)

            print(f"✅ Saved FullSC plot data: {file_path}")
                        
            plot_fullsc(
                sw_test_dates, y_test, preds_model, preds_pers27, preds_persN,
                delay=delay,
                fold_index=fold_index,
                corr_m=corr_m, p_m=p_m,
                save_path=os.path.join(save_path, f"FullSC_plot_fold{fold_index}_delay{delay}.png"),
                title_prefix="SpeedNet",
                model_type=model_type, three_column=True, gap_size=3, max_nans=5
            )


 
           # Attribution maps
            
            # print(f"X_test.shape: {X_test.shape}")
            # np.save(os.path.join(save_path, f"FullSC_fold{fold_index}_delay{delay}_X_test.npy"), X_test)
            
            # Splitted
            # split_and_save(X_test, loaded, model_filename, save_path)
            # print(loaded.summary())
            # save_attributions(X_test, loaded, save_path, prefix=f"FullSC_fold{fold_index}_delay{delay}")
            
            
            save_individual_attributions(
                    X_test=X_test,
                    sw_test_dates=sw_test_dates,
                    delay=delay,
                    model=loaded,
                    phase='FullSC',
                    base_save_dir=save_path
                    )





### SPEEDNET-EUV MODEL EVALUATION

In [ ]:
SpeedNet_EUV_evaluate = model_evaluate_and_persistence(
    sw_csv="../DATASET/Omni_velocity.csv",
    std_csv="../DATASET/standard_deviation_Velocity.csv",
    icme_excel="../DATASET/Original_Full_Richardson_cane_list(2010-2023).xlsx",
    img_path="../DATASET/stonyhurst_three_channel",
    model_path = "../DATASET/Results/SpeedNet_EUV/Full_SC/SpeedNet_EUV_ANALYSIS",
    delay = 3,
    fold_index = 4,
    batch_size=12,            # <- the known hyperparams you obtained
    lstm_unit=100,
    dense_unit=200,
    seed=42,
    test_fraction=0.2,        # same scheme used during training
    batch_window=20,
    save_path = "../DATASET/Results/SpeedNet_EUV/Full_SC/SpeedNet_EUV_evaluate/metrics",
    hsemetric=hsemetric,           # or pass your hsemetric object
    persistence_period_days=27.0,  # classic 27-day baseline: predicted(t)=V(t-27)
    persistence_fill=None,         # or "ffill"/"bfill" if you want daily filling
    N_period_days=27.0,            # N-day with delay: predicted(t)=V(t+delay-27). Change N as desired.
    model_type = "EUV"
)

Loaded 4510 SW records and 3427 images from /home/joseph/DL/Revised_SpeedNet/DATASET/stonyhurst_three_channel

=== Delay 3 ===
Total batches: 82, Test batches: 16, Train+Val batches: 66
Plotting during the Fold 4 and Delay: 3
Fold 4: Train 1060, Val 260, Test 320


2025-11-04 12:05:20.300849: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-04 12:05:20.994767: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:17:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-11-04 12:05:20.994841: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:73:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-11-04 12:05:20.994885: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:17:00.0/numa_node
Your kernel may h

10/10 [==============================] - 10s 21ms/step
✅ Saved FullSC plot data: ./SpeedNet_EUV_fullsc_data/SpeedNet_EUV_FullSC_fold4_delay3.pkl
Saved plot → /home/joseph/DL/Revised_SpeedNet/New/FULL_SC_ANALYSES/SpeedNet_EUV_evaluate/metrics/FullSC_plot_fold4_delay3_3col.png


### SPEEDNET-BM MODEL EVALUATION

In [ ]:
SpeedNet_BM_evaluate = model_evaluate_and_persistence(
    sw_csv="../DATASET/Omni_velocity.csv",
    std_csv="../DATASET/standard_deviation_Velocity.csv",
    icme_excel="../DATASET/Original_Full_Richardson_cane_list(2010-2023).xlsx",
    img_path="../DATASET/binary_map",
    model_path = "../DATASET/Results/SpeedNet_BM/Full_SC/SpeedNet_BM_ANALYSIS",
    delay = 3,
    fold_index = 0,
    batch_size=12,            # <- the known hyperparams you obtained
    lstm_unit=100,
    dense_unit=400,
    seed=42,
    test_fraction=0.2,        # same scheme used during training
    batch_window=20,
    save_path = "../DATASET/Results/SpeedNet_BM/Full_SC/SpeedNet_BM_evaluate/metrics",
    hsemetric=hsemetric,           # or pass your hsemetric object
    persistence_period_days=27.0,  # classic 27-day baseline: predicted(t)=V(t-27)
    persistence_fill=None,         # or "ffill"/"bfill" if you want daily filling
    N_period_days=27.0 ,            # N-day with delay: predicted(t)=V(t+delay-27). Change N as desired.
    model_type = "BM"
)



Loaded 4510 SW records and 3427 images from /home/joseph/DL/Revised_SpeedNet/DATASET/binary_map

=== Delay 3 ===
Total batches: 82, Test batches: 16, Train+Val batches: 66
Plotting during the Fold 0 and Delay: 3
Fold 0: Train 1060, Val 260, Test 320
10/10 [==============================] - 1s 17ms/step
✅ Saved FullSC plot data: ./SpeedNet_BM_fullsc_data/SpeedNet_BM_FullSC_fold0_delay3.pkl
Saved plot → /home/joseph/DL/Revised_SpeedNet/New/FULL_SC_ANALYSES/SpeedNet_BM_evaluate/metrics/FullSC_plot_fold0_delay3_3col.png
